# Phase 1 · 0: BPE tokenizer + tokenized corpus (run this ONCE)

## 0. Setup

In [ ]:
%pip install -q regex ijson

In [ ]:
import os, sys, json, time, glob, gzip, random, hashlib, heapq, shutil
import multiprocessing as mp
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

try:
    import regex as re_u
    HAVE_REGEX = True
except ImportError:
    import re as re_u
    HAVE_REGEX = False
import ijson

IS_KAGGLE = os.path.exists("/kaggle/working")
IS_COLAB = "google.colab" in sys.modules
if IS_KAGGLE:
    DEFAULT_WORK = "/kaggle/working/cs781"
elif IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_WORK = "/content/drive/MyDrive/cs781"
else:
    DEFAULT_WORK = str(Path.home() / "cs781")

WORK_DIR = Path(os.environ.get("CAPSTONE_WORK_DIR", DEFAULT_WORK))
ARTIFACTS = WORK_DIR / "artifacts"
TOK_DIR = ARTIFACTS / "tokenizers"
DATA_CACHE = ARTIFACTS / "tokenized"
for d in (TOK_DIR, DATA_CACHE):
    d.mkdir(parents=True, exist_ok=True)

def _scratch_dir():
    if os.environ.get("CAPSTONE_RAW_DIR"):
        return Path(os.environ["CAPSTONE_RAW_DIR"])
    if IS_KAGGLE or IS_COLAB:
        cands = [p for p in ("/kaggle/temp", "/tmp") if os.path.isdir(p) and os.access(p, os.W_OK)]
        best = max(cands, key=lambda p: shutil.disk_usage(p).free)
        return Path(best) / "cs781_raw"
    return WORK_DIR / "hf_data"
RAW_DIR = _scratch_dir()
RAW_DIR.mkdir(parents=True, exist_ok=True)

N_PROC = max(1, (os.cpu_count() or 2))
HAVE_FORK = "fork" in mp.get_all_start_methods()
print(f"work dir: {WORK_DIR} | raw dir: {RAW_DIR} (free {shutil.disk_usage(RAW_DIR).free/1e9:.0f} GB) | "
      f"cpus: {N_PROC} | regex: {HAVE_REGEX} | ijson backend: {ijson.backend}")
assert HAVE_REGEX, "install `regex` (pip install regex) - the stdlib fallback pattern is not Unicode-aware"

In [ ]:
SMOKE_TEST = os.environ.get("CAPSTONE_SMOKE", "0") == "1"

CFG = dict(
    seed=1337,

    data_dir=os.environ.get("CAPSTONE_DATA_DIR", ""),
    hf_repo="Exploration-Lab/CS-781-Capstone",
    hf_subdir="Phase_1_Pretraining",
    raw_access="auto",
    sources={
        "acts": "train_acts.json",
        "high_courts": "train_high_courts.json",
        "supreme_court": "train_supreme_court.json",
        "tribunals": "train_tribunals.json",
    },
    mix={"acts": 0.20, "supreme_court": 0.35, "high_courts": 0.25, "tribunals": 0.20},
    train_tokens=500_000_000,
    val_tokens=5_000_000,
    n_val_prompts=300,

    vocab_size=32_000,
    tok_train_chars=100_000_000,
    tok_min_word_freq=2,
    tok_split_digits=True,
)

if SMOKE_TEST:
    CFG.update(train_tokens=300_000, val_tokens=30_000, n_val_prompts=8,
               vocab_size=600, tok_train_chars=400_000, tok_min_word_freq=1)

assert abs(sum(CFG["mix"].values()) - 1.0) < 1e-6, "mix fractions must sum to 1"
assert CFG["vocab_size"] <= 65_536, "uint16 storage needs vocab_size <= 65,536"
random.seed(CFG["seed"]); np.random.seed(CFG["seed"])
print("smoke test:", SMOKE_TEST)

## 1. Raw data

In [ ]:
_HF_TOKEN, _HF_FS = None, None

def hf_token():
    global _HF_TOKEN
    if _HF_TOKEN is None:
        t = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
        if not t and IS_KAGGLE:
            try:
                from kaggle_secrets import UserSecretsClient
                t = UserSecretsClient().get_secret("HF_TOKEN")
            except Exception as e:
                print("[hf] no Kaggle secret HF_TOKEN:", type(e).__name__)
        if not t and IS_COLAB:
            try:
                from google.colab import userdata
                t = userdata.get("HF_TOKEN")
            except Exception:
                pass
        _HF_TOKEN = t or False
    return _HF_TOKEN or None

def hf_fs():
    global _HF_FS
    if _HF_FS is None:
        from huggingface_hub import HfFileSystem
        _HF_FS = HfFileSystem(token=hf_token())
    return _HF_FS

def _local_hit(rel):
    parts = rel.split("/")
    pats = ["/".join(parts[-2:])] + ([parts[-1]] if parts[-1].startswith("train_") else [])
    roots = [CFG["data_dir"]] if CFG["data_dir"] else []
    roots += ["/kaggle/input", str(RAW_DIR), str(WORK_DIR / "hf_data"), "/content"]
    for pat in pats:
        for r in roots:
            if r and os.path.isdir(r):
                hits = [h for h in glob.glob(f"{r}/**/{pat}", recursive=True) if os.path.getsize(h) > 0]
                if hits:
                    return hits[0]
    return None

_RESOLVED = {}
def resolve(rel):
    if rel in _RESOLVED:
        return _RESOLVED[rel]
    hit = _local_hit(rel)
    if hit:
        _RESOLVED[rel] = hit
        return hit
    from huggingface_hub import HfApi, hf_hub_download
    size = None
    try:
        info = HfApi(token=hf_token()).get_paths_info(CFG["hf_repo"], [rel], repo_type="dataset")
        size = info[0].size
    except Exception as e:
        raise RuntimeError(f"cannot reach {CFG['hf_repo']}/{rel} ({type(e).__name__}: {e}). The dataset is gated: "
                           "add an HF_TOKEN secret (Kaggle: Add-ons -> Secrets) or attach the raw files as a dataset.")
    free = shutil.disk_usage(RAW_DIR).free
    mode = CFG["raw_access"]
    if mode == "auto":
        mode = "download" if free > size + 3e9 else "stream"
    if mode == "download":
        print(f"[raw] downloading {rel} ({size/1e9:.2f} GB, {free/1e9:.0f} GB free) ...")
        t0 = time.time()
        p = hf_hub_download(CFG["hf_repo"], rel, repo_type="dataset", local_dir=str(RAW_DIR), token=hf_token())
        print(f"[raw]   done in {time.time()-t0:.0f}s -> {p}")
    else:
        p = f"hf://datasets/{CFG['hf_repo']}/{rel}"
        print(f"[raw] streaming {rel} ({size/1e9:.2f} GB) from Hugging Face")
    _RESOLVED[rel] = p
    return p

def source_path(name):
    return resolve(f"{CFG['hf_subdir']}/Train/{CFG['sources'][name]}")

In [ ]:
TEXT_KEYS = ("text", "content", "body", "judgment", "document")

def _extract_text(v):
    if isinstance(v, str):
        return v
    if isinstance(v, dict):
        for k in TEXT_KEYS:
            if isinstance(v.get(k), str):
                return v[k]
        strs = [x for x in v.values() if isinstance(x, str)]
        return max(strs, key=len) if strs else ""
    if isinstance(v, list):
        return "\n".join(x for x in v if isinstance(x, str))
    return ""

def _open_rb(path):
    path = str(path)
    if path.startswith("hf://"):
        return hf_fs().open(path[len("hf://"):], "rb", block_size=32 << 20)
    if path.endswith(".gz"):
        return gzip.open(path, "rb")
    return open(path, "rb")

def _sniff(path, cap=64 << 20):
    with _open_rb(path) as f:
        while True:
            c = f.read(1)
            if not c:
                return "empty"
            if not c.isspace():
                break
        if c == b"[":
            return "list"
        if c != b"{":
            return "jsonl"
        buf = bytearray(c)
        while b"\n" not in buf and len(buf) < cap:
            chunk = f.read(1 << 20)
            if not chunk:
                break
            buf += chunk
        if b"\n" not in buf:
            return "dict"
        line, rest = bytes(buf).split(b"\n", 1)
        try:
            obj = json.loads(line)
        except ValueError:
            return "dict"
        more = rest.strip() or f.read(1 << 16).strip()
        return "jsonl" if isinstance(obj, dict) and more else "dict"

def _iter_lines(f, chunk=8 << 20):
    buf = b""
    while True:
        data = f.read(chunk)
        if not data:
            break
        buf += data
        *lines, buf = buf.split(b"\n")
        yield from lines
    if buf:
        yield buf

def iter_docs(path):
    kind = _sniff(path)
    with _open_rb(path) as f:
        if kind == "dict":
            for k, v in ijson.kvitems(f, ""):
                yield str(k), _extract_text(v)
        elif kind == "list":
            for i, v in enumerate(ijson.items(f, "item")):
                yield (str(v.get("id", i)) if isinstance(v, dict) else str(i)), _extract_text(v)
        elif kind == "jsonl":
            for i, line in enumerate(_iter_lines(f)):
                line = line.strip()
                if line:
                    v = json.loads(line)
                    yield (str(v.get("id", i)) if isinstance(v, dict) else str(i)), _extract_text(v)

def split_of(source, doc_id, val_rate):
    h = int(hashlib.md5(f"{source}:{doc_id}".encode()).hexdigest()[:8], 16) / 2**32
    return "val" if h < val_rate else "train"

VAL_RATE = min(0.2, max(0.01, 1.5 * CFG["val_tokens"] / max(1, CFG["train_tokens"])))
print("val doc rate:", round(VAL_RATE, 4))

In [ ]:
for s, frac in CFG["mix"].items():
    if frac <= 0:
        continue
    _id, _t = next(iter_docs(source_path(s)))
    print(f"  {s:14s} first doc id={_id!r} chars={len(_t)} :: {_t[:110]!r}")

## 2. Byte-level BPE tokenizer (from scratch)

In [ ]:
SPECIAL_TOKENS = ["<|endoftext|>", "<|pad|>", "<|bos|>", "<|sep|>", "<|mask|>"]

def make_pattern(split_digits=True):
    num = r"\p{N}{1,3}" if split_digits else r"\p{N}+"
    return (r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?""" + num +
            r"""| ?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+""")

def _merge_ids(ids, pair, new_id):
    out, i, n = [], 0, len(ids)
    a, b = pair
    while i < n:
        if i < n - 1 and ids[i] == a and ids[i + 1] == b:
            out.append(new_id); i += 2
        else:
            out.append(ids[i]); i += 1
    return out

_COUNT_PAT = None
def _count_words(texts_and_pattern):
    texts, pattern = texts_and_pattern
    global _COUNT_PAT
    if _COUNT_PAT is None or _COUNT_PAT.pattern != pattern:
        _COUNT_PAT = re_u.compile(pattern)
    c = Counter()
    for t in texts:
        c.update(_COUNT_PAT.findall(t))
    return c

In [ ]:
class BPETokenizer:
    def __init__(self, merges=None, pattern=None, specials=SPECIAL_TOKENS):
        self.pattern = pattern or make_pattern()
        self.pat = re_u.compile(self.pattern)
        self.merges = [tuple(m) for m in (merges or [])]
        self.specials = list(specials)
        self._build()

    def _build(self):
        self.ranks = {p: i for i, p in enumerate(self.merges)}
        self.vocab = {i: bytes([i]) for i in range(256)}
        for i, (a, b) in enumerate(self.merges):
            self.vocab[256 + i] = self.vocab[a] + self.vocab[b]
        base = 256 + len(self.merges)
        self.n_regular = base
        self.special_to_id = {s: base + i for i, s in enumerate(self.specials)}
        self.id_to_special = {v: k for k, v in self.special_to_id.items()}
        self.vocab_size = base + len(self.specials)
        self._special_re = re_u.compile("(" + "|".join(re_u.escape(s) for s in self.specials) + ")")
        self.cache = {}

    def __len__(self):
        return self.vocab_size

    @property
    def eot_id(self): return self.special_to_id["<|endoftext|>"]
    @property
    def pad_id(self): return self.special_to_id["<|pad|>"]
    @property
    def bos_id(self): return self.special_to_id["<|bos|>"]
    @property
    def sep_id(self): return self.special_to_id["<|sep|>"]
    @property
    def mask_id(self): return self.special_to_id["<|mask|>"]

    @classmethod
    def train(cls, text_iter, vocab_size, pattern, min_word_freq=2, n_proc=1, verbose=True):
        t0 = time.time()
        def batches(it, size=200):
            buf = []
            for t in it:
                buf.append(t)
                if len(buf) == size:
                    yield (buf, pattern); buf = []
            if buf:
                yield (buf, pattern)
        counts = Counter()
        if n_proc > 1 and HAVE_FORK:
            with mp.get_context("fork").Pool(n_proc) as pool:
                for c in pool.imap_unordered(_count_words, batches(text_iter), chunksize=1):
                    counts.update(c)
        else:
            for b in batches(text_iter):
                counts.update(_count_words(b))
        words, freqs = [], []
        for w, c in counts.items():
            if c >= min_word_freq:
                words.append(list(w.encode("utf-8"))); freqs.append(c)
        del counts
        if verbose:
            print(f"[bpe] {len(words):,} word types after min_freq={min_word_freq} ({time.time()-t0:.0f}s)")

        pair_counts = defaultdict(int)
        where = defaultdict(list)
        for i, w in enumerate(words):
            f = freqs[i]
            seen = set()
            for p in zip(w, w[1:]):
                pair_counts[p] += f
                if p not in seen:
                    where[p].append(i); seen.add(p)
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        n_merges = vocab_size - 256 - len(SPECIAL_TOKENS)
        assert n_merges > 0, "vocab_size too small"
        merges = []
        while len(merges) < n_merges and heap:
            negc, p = heapq.heappop(heap)
            cur = pair_counts.get(p, 0)
            if cur <= 0 or -negc != cur:
                continue
            new_id = 256 + len(merges)
            merges.append(p)
            touched = set()
            for i in set(where.pop(p, ())):
                w = words[i]
                nw = _merge_ids(w, p, new_id)
                if len(nw) == len(w):
                    continue
                f = freqs[i]
                old_pc = Counter(zip(w, w[1:]))
                new_pc = Counter(zip(nw, nw[1:]))
                for q, n in old_pc.items():
                    pair_counts[q] -= n * f; touched.add(q)
                for q, n in new_pc.items():
                    pair_counts[q] += n * f; touched.add(q)
                    if q not in old_pc:
                        where[q].append(i)
                words[i] = nw
            for q in touched:
                c = pair_counts.get(q, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, q))
                else:
                    pair_counts.pop(q, None)
            if verbose and len(merges) % 2000 == 0:
                print(f"[bpe] {len(merges):,}/{n_merges:,} merges ({time.time()-t0:.0f}s)")
        if verbose:
            print(f"[bpe] done: {len(merges):,} merges in {time.time()-t0:.0f}s")
        if len(merges) < n_merges:
            print(f"[bpe] warning: only {len(merges):,}/{n_merges:,} merges possible -> vocab is smaller than requested "
                  "(raise tok_train_chars or lower tok_min_word_freq)")
        return cls(merges=merges, pattern=pattern)

    def _encode_word(self, s):
        hit = self.cache.get(s)
        if hit is not None:
            return hit
        ids = list(s.encode("utf-8"))
        ranks = self.ranks
        while len(ids) >= 2:
            best, best_r = None, None
            for p in zip(ids, ids[1:]):
                r = ranks.get(p)
                if r is not None and (best_r is None or r < best_r):
                    best, best_r = p, r
            if best is None:
                break
            ids = _merge_ids(ids, best, 256 + best_r)
        if len(self.cache) > 500_000:
            self.cache.clear()
        self.cache[s] = ids
        return ids

    def encode_ordinary(self, text):
        out = []
        for w in self.pat.findall(text):
            out.extend(self._encode_word(w))
        return out

    def encode(self, text, allow_special=True):
        if not allow_special:
            return self.encode_ordinary(text)
        out = []
        for part in self._special_re.split(text):
            if part in self.special_to_id:
                out.append(self.special_to_id[part])
            elif part:
                out.extend(self.encode_ordinary(part))
        return out

    def decode(self, ids, skip_special=True):
        buf, out = [], []
        for i in ids:
            i = int(i)
            if i in self.id_to_special:
                if buf:
                    out.append(b"".join(buf).decode("utf-8", errors="replace")); buf = []
                if not skip_special:
                    out.append(self.id_to_special[i])
            else:
                buf.append(self.vocab[i])
        if buf:
            out.append(b"".join(buf).decode("utf-8", errors="replace"))
        return "".join(out)

    def save(self, path, meta=None):
        path = Path(path)
        tmp = path.with_suffix(".tmp")
        json.dump({"type": "byte_bpe", "pattern": self.pattern, "specials": self.specials,
                   "merges": self.merges, "vocab_size": self.vocab_size, "meta": meta or {}},
                  open(tmp, "w", encoding="utf-8"))
        os.replace(tmp, path)

    @classmethod
    def load(cls, path):
        d = json.load(open(path, encoding="utf-8"))
        return cls(merges=d["merges"], pattern=d["pattern"], specials=d["specials"])

## 3. Train (or load) the tokenizer

In [ ]:
def _hash(obj):
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:10]

TOK_SPEC = dict(v=1, vocab_size=CFG["vocab_size"], pattern=make_pattern(CFG["tok_split_digits"]),
                specials=SPECIAL_TOKENS, mix=CFG["mix"], chars=CFG["tok_train_chars"],
                min_freq=CFG["tok_min_word_freq"], val_rate=round(VAL_RATE, 6), smoke=SMOKE_TEST)
TOK_NAME = f"bpe_v{CFG['vocab_size']}_{_hash(TOK_SPEC)}.json"

def find_cached(name, extra_roots=("/kaggle/input",)):
    for root in [TOK_DIR, DATA_CACHE, *extra_roots]:
        hits = glob.glob(f"{root}/**/{name}", recursive=True)
        if hits:
            return Path(hits[0])
    return None

def tokenizer_training_texts():
    for s, frac in CFG["mix"].items():
        if frac <= 0:
            continue
        quota, got = int(CFG["tok_train_chars"] * frac), 0
        for doc_id, text in iter_docs(source_path(s)):
            if not text or split_of(s, doc_id, VAL_RATE) != "train":
                continue
            yield text
            got += len(text)
            if got >= quota:
                break
        print(f"[bpe] {s}: {got:,} chars")

def get_or_train_tokenizer():
    cached = find_cached(TOK_NAME)
    if cached:
        print("[bpe] loading cached tokenizer:", cached)
        return BPETokenizer.load(cached), Path(cached)
    print("[bpe] no cached tokenizer found -> training", TOK_NAME)
    tok = BPETokenizer.train(tokenizer_training_texts(), CFG["vocab_size"], TOK_SPEC["pattern"],
                             min_word_freq=CFG["tok_min_word_freq"], n_proc=N_PROC)
    path = TOK_DIR / TOK_NAME
    tok.save(path, meta=TOK_SPEC)
    print("[bpe] saved ->", path)
    return tok, path

In [ ]:
tok, TOK_PATH = get_or_train_tokenizer()
VOCAB_SIZE = tok.vocab_size
print("vocab size:", VOCAB_SIZE, "| eot/pad/bos/sep/mask:", tok.eot_id, tok.pad_id, tok.bos_id, tok.sep_id, tok.mask_id)

In [ ]:
for s, frac in CFG["mix"].items():
    if frac <= 0:
        continue
    for doc_id, text in iter_docs(source_path(s)):
        if text and split_of(s, doc_id, VAL_RATE) == "val":
            text = text[:20000]
            ids = tok.encode(text)
            assert tok.decode(ids) == text, "round-trip failed"
            print(f"  {s:14s} fertility={len(ids)/max(1,len(text.split())):.3f} tok/word  chars/tok={len(text)/max(1,len(ids)):.2f}")
            break
print("example:", [tok.decode([i]) for i in tok.encode("The appellant filed a writ petition under Article 226 of the Constitution.")])

## 4. Tokenize the corpus

In [ ]:
DATA_SPEC = dict(v=2, tok=TOK_NAME, mix=CFG["mix"], train_tokens=CFG["train_tokens"],
                 val_tokens=CFG["val_tokens"], n_val_prompts=CFG["n_val_prompts"], val_rate=round(VAL_RATE, 6),
                 smoke=SMOKE_TEST)
DATA_NAME = f"data_{_hash(DATA_SPEC)}"
NP_DTYPE = np.uint16 if VOCAB_SIZE <= 2**16 else np.uint32

_W_TOK = None
def _w_init(tok_path):
    global _W_TOK
    _W_TOK = BPETokenizer.load(tok_path)

def _w_encode(item):
    split, text = item
    ids = _W_TOK.encode_ordinary(text)
    ids.append(_W_TOK.eot_id)
    return split, np.asarray(ids, dtype=NP_DTYPE)

def build_tokenized_corpus():
    marker = find_cached(f"{DATA_NAME}/meta.json")
    if marker:
        print("[data] using cached tokenized corpus:", marker.parent)
        return marker.parent
    out = DATA_CACHE / DATA_NAME
    out.mkdir(parents=True, exist_ok=True)
    f = {sp: open(out / f"{sp}.bin", "wb") for sp in ("train", "val")}
    total = {"train": 0, "val": 0}
    per_source = {}
    rng = random.Random(CFG["seed"])
    val_prompts = []
    t0 = time.time()
    for s, frac in CFG["mix"].items():
        if frac <= 0:
            continue
        quota = {"train": int(CFG["train_tokens"] * frac), "val": int(CFG["val_tokens"] * frac)}
        n_prompts = max(1, round(CFG["n_val_prompts"] * frac))
        got = {"train": 0, "val": 0}
        n_src_prompts = [0]

        def doc_stream():
            for doc_id, text in iter_docs(source_path(s)):
                if not text:
                    continue
                sp = split_of(s, doc_id, VAL_RATE)
                if got[sp] >= quota[sp]:
                    if all(got[k] >= quota[k] for k in got):
                        return
                    continue
                if sp == "val" and n_src_prompts[0] < n_prompts:
                    w = text.split()
                    if len(w) >= 512 + 128 + 10:
                        st = rng.randint(0, len(w) - 640)
                        val_prompts.append({"source": s, "doc_id": doc_id,
                                            "prompt": " ".join(w[st:st + 512]),
                                            "reference": " ".join(w[st + 512:st + 640])})
                        n_src_prompts[0] += 1
                yield sp, text

        if HAVE_FORK and N_PROC > 1:
            pool = mp.get_context("fork").Pool(N_PROC, initializer=_w_init, initargs=(str(TOK_PATH),))
            encoded = pool.imap(_w_encode, doc_stream(), chunksize=8)
        else:
            pool = None
            _w_init(str(TOK_PATH))
            encoded = map(_w_encode, doc_stream())
        last_print = time.time()
        try:
            for sp, arr in encoded:
                if got[sp] >= quota[sp]:
                    continue
                f[sp].write(arr.tobytes())
                got[sp] += len(arr); total[sp] += len(arr)
                if time.time() - last_print > 120:
                    last_print = time.time()
                    print(f"[data]   {s}: train {got['train']/1e6:.1f}M/{quota['train']/1e6:.0f}M  "
                          f"val {got['val']/1e6:.2f}M/{quota['val']/1e6:.1f}M  ({time.time()-t0:.0f}s)")
                if all(got[k] >= quota[k] for k in got):
                    break
        finally:
            if pool is not None:
                pool.terminate()
        per_source[s] = dict(got)
        for sp in got:
            if got[sp] < 0.95 * quota[sp]:
                print(f"[data] WARNING: {s} {sp} has only {got[sp]:,}/{quota[sp]:,} tokens -> the source ran out; "
                      "the realised mix differs from CFG['mix'] (lower its share)")
        print(f"[data] {s:14s} train={got['train']:,} val={got['val']:,} prompts={n_src_prompts[0]}  ({time.time()-t0:.0f}s)")
    for fh in f.values():
        fh.close()

    shutil.copy(TOK_PATH, out / "tokenizer.json")
    test_rel = f"{CFG['hf_subdir']}/Test/test.json"
    try:
        test_src = resolve(test_rel)
        if test_src.startswith("hf://"):
            from huggingface_hub import hf_hub_download
            test_src = hf_hub_download(CFG["hf_repo"], test_rel, repo_type="dataset",
                                       local_dir=str(RAW_DIR), token=hf_token())
        shutil.copy(test_src, out / "test.json")
        test_file = "test.json"
        n_test = len(json.load(open(out / "test.json", encoding="utf-8")))
        print(f"[data] copied official test prompts: {n_test}")
    except Exception as e:
        test_file = None
        print(f"[data] WARNING: could not copy Test/test.json ({type(e).__name__}: {e}) - submissions will not work")

    json.dump(val_prompts, open(out / "val_prompts.json", "w", encoding="utf-8"))
    json.dump({"spec": DATA_SPEC, "dtype": np.dtype(NP_DTYPE).name, "total": total, "per_source": per_source,
               "n_val_prompts": len(val_prompts), "vocab_size": VOCAB_SIZE, "tokenizer_file": "tokenizer.json",
               "tokenizer_name": TOK_NAME, "test_file": test_file, "smoke": SMOKE_TEST, "created": time.time()},
              open(out / "meta.json", "w", encoding="utf-8"), indent=2)
    print(f"[data] done in {time.time()-t0:.0f}s")
    return out

In [ ]:
DATA_PATH = build_tokenized_corpus()
META = json.load(open(DATA_PATH / "meta.json", encoding="utf-8"))
print(json.dumps(META["total"]), "| per source:", json.dumps(META["per_source"]), "| val prompts:", META["n_val_prompts"])

## 5. Verify the output

In [ ]:
def load_split(split):
    return np.memmap(DATA_PATH / f"{split}.bin", dtype=np.dtype(META["dtype"]), mode="r")

tok2 = BPETokenizer.load(DATA_PATH / "tokenizer.json")
assert tok2.vocab_size == META["vocab_size"] == VOCAB_SIZE
for split in ("train", "val"):
    arr = load_split(split)
    assert len(arr) == META["total"][split], f"{split}.bin size mismatch"
    assert int(arr[: 5_000_000].max()) < VOCAB_SIZE
    print(f"{split}: {len(arr):,} tokens | {int((arr[:5_000_000] == tok2.eot_id).sum())} docs in the first 5M tokens")
print("decoded sample:", repr(tok2.decode(load_split("train")[1000:1060].tolist())))
vp = json.load(open(DATA_PATH / "val_prompts.json", encoding="utf-8"))
assert all(len(v["prompt"].split()) == 512 and len(v["reference"].split()) == 128 for v in vp)
print("val prompts by source:", dict(Counter(v["source"] for v in vp)))
if META["test_file"]:
    t = json.load(open(DATA_PATH / META["test_file"], encoding="utf-8"))
    k0 = next(iter(t))
    print(f"test prompts: {len(t)} | id {k0!r}: {len(t[k0]['text'].split())} words")

print("\nfolder contents:")
for p in sorted(DATA_PATH.iterdir()):
    print(f"  {p.name:20s} {p.stat().st_size/1e6:10.1f} MB")
print(f"\nDATA_NAME = {DATA_NAME!r}   <- the model notebooks pick this up automatically "
      "(or pin it with CFG['data_name'])")

## Next steps